# MVP: target_role -> role_profile -> hybrid search -> reranker

Этот ноутбук запускает локальный pipeline по SQLite-базе `data/app.db` и сохраняет результаты в `outputs/role_relevance_product_analyst.json` и `outputs/role_relevance_product_analyst.xlsx`.

По умолчанию используется rule-based fallback reranker, поэтому API-ключ не нужен.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'app').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

In [ ]:
from app.db import SessionLocal
from app.services.role_relevance_pipeline import export_role_relevance_result, run_role_relevance_pipeline

target_role = 'Продуктовый аналитик'
limit = 300
use_llm_reranker = False  # True будет пробовать LLM только если OPENAI_ENABLED=true

db = SessionLocal()
try:
    result = run_role_relevance_pipeline(
        db,
        target_role=target_role,
        limit=limit,
        use_llm_reranker=use_llm_reranker,
    )
finally:
    db.close()

result.stats.model_dump()

In [ ]:
result.role_profile.model_dump()

In [ ]:
preview = [item.model_dump() for item in result.results[:20]]
preview

In [ ]:
json_path, xlsx_path = export_role_relevance_result(
    result,
    output_dir=PROJECT_ROOT / 'outputs',
    basename='role_relevance_product_analyst',
)
json_path, xlsx_path

In [ ]:
# Быстрая проверка сегментов
from collections import Counter

Counter(item.market_segment for item in result.results)